# Day 1: Representation as Measurement

The goal today is to see how text becomes data. The same corpus can support different conclusions depending on the unit of analysis, preprocessing, feature weighting, and similarity measure.

By the end you should be able to:

1. Build a document-term matrix.
2. Compare raw counts, binary counts, normalized counts, and TF-IDF.
3. Explain what common preprocessing choices remove or preserve.
4. Compute document similarity.
5. Compare groups using distinctive words.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 120)

In [ ]:
from pathlib import Path


def find_data_dir():
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')


DATA_DIR = find_data_dir()
DATA_DIR

## spaCy setup

The notebooks use spaCy for tokenization and preprocessing. If `en_core_web_sm` is installed, the same setup also shows POS tags, lemmas, and dependency parses. If not, the notebooks still run with a blank English tokenizer.

In [ ]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
    print('Loaded spaCy model: en_core_web_sm')
except OSError:
    nlp = spacy.blank('en')
    nlp.add_pipe('sentencizer')
    print('Using spacy.blank("en"). Install en_core_web_sm for POS tags, lemmas, and dependency parses.')


def spacy_preprocess(text, remove_stop=True, keep_alpha=True, min_len=2):
    """Tokenize and lightly preprocess text with spaCy."""
    doc = nlp.make_doc(str(text))
    tokens = []
    for token in doc:
        if keep_alpha and not token.is_alpha:
            continue
        if remove_stop and token.is_stop:
            continue
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


def spacy_analyzer(text):
    return spacy_preprocess(text)


def spacy_analyzer_with_bigrams(text):
    tokens = spacy_preprocess(text)
    bigrams = [tokens[i] + '_' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams


def spacy_token_table(text):
    """Show tokenization, preprocessing flags, and parses when a full model is available."""
    doc = nlp(str(text))
    rows = []
    for token in doc:
        rows.append({
            'text': token.text,
            'lemma': token.lemma_ or '(model needed)',
            'pos': token.pos_ or '(model needed)',
            'dep': token.dep_ or '(model needed)',
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'kept_by_preprocess': token.text.lower() in spacy_preprocess(token.text, remove_stop=True)
        })
    return pd.DataFrame(rows)

## 1. A tiny corpus

Start with a toy example where every transformation is visible. This is not realistic, but it makes the representation choices inspectable.

In [ ]:
toy_texts = [
    'John loves ice cream and oranges.',
    'Mary loves chocolate ice cream.',
    'John dislikes chocolate but loves oranges.',
    'Mary hates oranges and loves chocolate.'
]

toy = pd.DataFrame({
    'doc_id': ['doc_1', 'doc_2', 'doc_3', 'doc_4'],
    'speaker': ['John', 'Mary', 'John', 'Mary'],
    'text': toy_texts
})
toy

## 2. spaCy tokenization and parsing

This table shows what spaCy sees in one sentence. With `en_core_web_sm`, the lemma, POS, and dependency columns become real parses; with the fallback tokenizer, the tokenization and stop-word flags still work.

In [ ]:
spacy_token_table(toy_texts[0])

In [ ]:
vectorizer = CountVectorizer(analyzer=spacy_analyzer)
X = vectorizer.fit_transform(toy['text'])

dtm = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(), index=toy['doc_id'])
dtm

## 2. Representation choices

Compare several common representations. Ask what each one makes easy to see and what each one hides.

### Methodology formulas: representation as measurement

Most of the notebook turns raw text into a document-feature matrix. If document $d$ contains vocabulary item $v$, the count representation is

$$X_{dv} = c(v, d).$$

TF-IDF keeps local frequency but downweights terms that appear in many documents:

$$\mathrm{tfidf}_{dv} = \mathrm{tf}_{dv} \times \log\left(\frac{N}{\mathrm{df}_v}\right),$$

where $N$ is the number of documents and $\mathrm{df}_v$ is the number of documents containing term $v$. Similarity between two documents is usually measured by cosine similarity:

$$\cos(x_i, x_j) = \frac{x_i^\top x_j}{\lVert x_i \rVert_2\lVert x_j \rVert_2}.$$

For distinctive terms by group, a simple log-ratio compares relative term rates:

$$\delta_v = \log\left(\frac{c_{v,A}+\alpha}{\sum_j c_{j,A}+\alpha V}\right) - \log\left(\frac{c_{v,B}+\alpha}{\sum_j c_{j,B}+\alpha V}\right).$$

The two-dimensional document map later in the notebook uses a low-rank approximation of the feature matrix:

$$X \approx U_k \Sigma_k V_k^\top.$$


In [ ]:
representations = {
    'spaCy tokens, raw counts': CountVectorizer(
        analyzer=lambda text: spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1)
    ),
    'spaCy tokens, binary word presence': CountVectorizer(
        analyzer=lambda text: spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1),
        binary=True
    ),
    'spaCy preprocessing: no stop words': CountVectorizer(analyzer=spacy_analyzer),
    'spaCy preprocessing + TF-IDF': TfidfVectorizer(analyzer=spacy_analyzer)
}

for name, vec in representations.items():
    matrix = vec.fit_transform(toy['text'])
    table = pd.DataFrame(matrix.toarray(), columns=vec.get_feature_names_out(), index=toy['doc_id'])
    print()
    print(name.upper())
    display(table.round(3))

## 3. Move to real data

We will use State of the Union speeches. The file has one row per speech, with president, year, party, and text.

In [ ]:
sotu = pd.read_csv(DATA_DIR / 'sotu.csv')
sotu['year_numeric'] = pd.to_numeric(sotu['year'], errors='coerce')

sotu[['president', 'year', 'party', 'sotu_type', 'text']].head()

In [ ]:
modern = (
    sotu
    .dropna(subset=['year_numeric', 'party', 'text'])
    .query("party in ['Democratic', 'Republican'] and year_numeric >= 1950")
    .copy()
)

modern['doc_label'] = modern['president'] + ' (' + modern['year'].astype(str) + ')'
modern[['doc_label', 'party', 'sotu_type']].head(), modern.shape

## 4. Most frequent terms

Frequency is simple and useful, but frequent words are not always the most substantively interesting words.

In [ ]:
count_vec = CountVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=2000)
counts = count_vec.fit_transform(modern['text'])
terms = count_vec.get_feature_names_out()

term_counts = pd.Series(np.asarray(counts.sum(axis=0)).ravel(), index=terms)
term_counts.sort_values(ascending=False).head(20).to_frame('count')

In [ ]:
term_counts.sort_values(ascending=False).head(20).sort_values().plot(kind='barh', figsize=(7, 5))
plt.title('Most frequent terms in modern SOTU speeches')
plt.xlabel('Count')
plt.tight_layout()

## 4a. Frequency structure and document length

Advanced text analysis often starts with diagnostics. Zipf-style frequency plots and document-length distributions reveal whether a corpus is dominated by a few common terms or by uneven document sizes.

In [ ]:
modern['n_spacy_tokens'] = modern['text'].apply(
    lambda text: len(spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ranked_counts = term_counts.sort_values(ascending=False).reset_index(drop=True)
axes[0].loglog(np.arange(1, len(ranked_counts) + 1), ranked_counts.values)
axes[0].set_title('Zipf-style term frequency plot')
axes[0].set_xlabel('Term rank')
axes[0].set_ylabel('Frequency')

sns.histplot(data=modern, x='n_spacy_tokens', hue='party', bins=25, element='step', ax=axes[1])
axes[1].set_title('Document length by party')
axes[1].set_xlabel('Number of spaCy tokens')

plt.tight_layout()

### Additional demo: sparsity and vocabulary growth

Document-term matrices are mostly zeros. This demo makes that sparsity visible and shows how the observed vocabulary grows as we add speeches in chronological order.

In [ ]:
matrix_cells = counts.shape[0] * counts.shape[1]
sparsity = 1 - counts.nnz / matrix_cells

print(f'Document-term matrix shape: {counts.shape[0]} documents x {counts.shape[1]} terms')
print(f'Nonzero entries: {counts.nnz:,} of {matrix_cells:,}')
print(f'Sparsity: {sparsity:.1%}')

sample_rows = np.linspace(0, counts.shape[0] - 1, min(40, counts.shape[0]), dtype=int)
sample_cols = np.asarray(counts.sum(axis=0)).ravel().argsort()[::-1][:80]
sparsity_view = (counts[sample_rows][:, sample_cols] > 0).toarray()
visible_terms = terms[sample_cols]

seen_terms = set()
growth_rows = []
for order, (_, row) in enumerate(modern.sort_values('year_numeric').iterrows(), start=1):
    seen_terms.update(spacy_preprocess(row['text']))
    growth_rows.append({
        'speech_number': order,
        'year': row['year_numeric'],
        'vocabulary_size': len(seen_terms)
    })

vocab_growth = pd.DataFrame(growth_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(
    sparsity_view,
    cmap=['white', '#4C72B0'],
    cbar=False,
    xticklabels=visible_terms,
    yticklabels=False,
    ax=axes[0]
)
axes[0].set_title('Nonzero entries for top terms')
axes[0].set_xlabel('Terms ordered by corpus frequency')
axes[0].set_ylabel('Sampled documents')
axes[0].tick_params(axis='x', labelrotation=90, labelsize=7)

sns.lineplot(data=vocab_growth, x='year', y='vocabulary_size', marker='o', ax=axes[1])
axes[1].set_title('Vocabulary growth over SOTU speeches')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Cumulative unique preprocessed terms')

plt.tight_layout()

## 5. Distinctive terms by group

Here we compare Democratic and Republican speeches using a smoothed log ratio. Positive values are more Democratic in this coding; negative values are more Republican.

In [ ]:
count_table = pd.DataFrame(counts.toarray(), columns=terms)
count_table['_party_label'] = modern['party'].to_numpy()

group_counts = count_table.groupby('_party_label')[list(terms)].sum()
group_counts = group_counts.loc[['Democratic', 'Republican']]

vocab_size = len(terms)
dem_rate = (group_counts.loc['Democratic'] + 1) / (group_counts.loc['Democratic'].sum() + vocab_size)
rep_rate = (group_counts.loc['Republican'] + 1) / (group_counts.loc['Republican'].sum() + vocab_size)

log_ratio = np.log(dem_rate) - np.log(rep_rate)
distinctive = pd.DataFrame({'term': terms, 'dem_vs_rep_log_ratio': log_ratio.values})

pd.concat([
    distinctive.sort_values('dem_vs_rep_log_ratio', ascending=False).head(12),
    distinctive.sort_values('dem_vs_rep_log_ratio', ascending=True).head(12)
])

## 5a. Visualizing distinctive terms

The plot below makes the group contrast easier to audit than a table. Large positive values are more Democratic; large negative values are more Republican.

In [ ]:
top_dem = distinctive.sort_values('dem_vs_rep_log_ratio', ascending=False).head(12)
top_rep = distinctive.sort_values('dem_vs_rep_log_ratio', ascending=True).head(12)
plot_terms = pd.concat([top_rep, top_dem]).sort_values('dem_vs_rep_log_ratio')

plt.figure(figsize=(8, 7))
colors = np.where(plot_terms['dem_vs_rep_log_ratio'] > 0, '#4C72B0', '#C44E52')
plt.barh(plot_terms['term'], plot_terms['dem_vs_rep_log_ratio'], color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Distinctive terms: Democratic vs. Republican SOTU speeches')
plt.xlabel('Smoothed log ratio')
plt.tight_layout()

## 6. Document similarity

A vector space lets us ask which speeches are similar. Similarity is a property of the representation, not just the original texts.

In [ ]:
tfidf_vec = TfidfVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=3000)
tfidf = tfidf_vec.fit_transform(modern['text'])
similarity = cosine_similarity(tfidf)

pairs = []
labels = modern['doc_label'].to_list()
parties = modern['party'].to_list()

for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        pairs.append({
            'doc_1': labels[i],
            'party_1': parties[i],
            'doc_2': labels[j],
            'party_2': parties[j],
            'cosine_similarity': similarity[i, j]
        })

pd.DataFrame(pairs).sort_values('cosine_similarity', ascending=False).head(10)

## 6a. Similarity heatmap

Similarity matrices are useful for checking whether documents cluster by time, speaker, party, genre, or some artifact of preprocessing.

In [ ]:
recent = modern.sort_values('year_numeric').tail(16).copy()
recent_tfidf = tfidf_vec.transform(recent['text'])
recent_similarity = cosine_similarity(recent_tfidf)

plt.figure(figsize=(10, 8))
sns.heatmap(
    recent_similarity,
    xticklabels=recent['doc_label'],
    yticklabels=recent['doc_label'],
    cmap='viridis',
    vmin=0,
    vmax=1
)
plt.title('Cosine similarity among recent SOTU speeches')
plt.xticks(rotation=80, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

## 6b. Representation geometry

A two-dimensional projection is not a model result by itself. It is a diagnostic view of how the chosen representation arranges documents.

In [ ]:
svd2 = TruncatedSVD(n_components=2, random_state=42)
coords = svd2.fit_transform(tfidf)

plot_df = modern[['president', 'year_numeric', 'party', 'sotu_type']].copy()
plot_df['x'] = coords[:, 0]
plot_df['y'] = coords[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x='x', y='y', hue='party', style='sotu_type', alpha=0.85)
plt.title('SVD projection of TF-IDF document vectors')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.tight_layout()

## 7. Student task

Pick one change below and rerun the analysis:

- Change `min_df` from 2 to 5.
- Use binary word presence instead of counts.
- Include bigrams by using `spacy_analyzer_with_bigrams` as the vectorizer analyzer.
- Compare speeches before and after 1980.

Write three sentences: what changed, why did it change, and which representation best fits the research question?

In [ ]:
# Your turn: modify the vectorizer or the grouping variable here.
student_vec = CountVectorizer(analyzer=spacy_analyzer, min_df=2)
# For bigrams, try:
# student_vec = CountVectorizer(analyzer=spacy_analyzer_with_bigrams, min_df=2)

student_X = student_vec.fit_transform(modern['text'])

print(student_X.shape)
pd.Series(
    np.asarray(student_X.sum(axis=0)).ravel(),
    index=student_vec.get_feature_names_out()
).sort_values(ascending=False).head(20)